In [ ]:
from google.colab import drive; import os; drive.mount('/content/drive'); os.makedirs('/content/drive/MyDrive/space_robotics', exist_ok=True)


Mounted at /content/drive


In [ ]:
from tqdm.auto import tqdm
import time
import sys
import numpy as np
import pandas as pd

In [ ]:
import time
import sys
import numpy as np

app = None
try:
    import omni.kit.app
    app = omni.kit.app.get_app()
except Exception:
    app = None

# Constants
G = 6.67430e-11
M_earth = 5.972e24
R_earth = 6.371e6
h_sat = 500e3  # satellite height in meters

r_sat = R_earth + h_sat
T_sat = 2 * np.pi * np.sqrt(r_sat**3 / (G * M_earth))
omega_sat = 2 * np.pi / T_sat  # satellite angular velocity in rad/s

# Basis vectors
i_cap = np.array([1.0, 0.0, 0.0])
j_cap = np.array([0.0, 1.0, 0.0])
k_cap = np.array([0.0, 0.0, 1.0])


# --- helper functions ---
def rotate_vector_around_axis_by_angle(vector, axis, angle_rad):
    vector = np.array(vector, float)
    axis = np.array(axis, float)
    axis /= np.linalg.norm(axis)
    t = angle_rad
    rotated = vector * np.cos(t) + np.cross(axis, vector) * np.sin(t) + axis * np.dot(axis, vector) * (1 - np.cos(t))
    rotated /= np.linalg.norm(rotated)
    return rotated


def create_circular_trajectory_parameters_for_rectannas(lat_rad=0.0, long_rad=0.0, radius_earth=R_earth):
    """
    Rectenna on Earth. All angles in radians.
    """
    center = radius_earth * np.sin(lat_rad) * k_cap
    radius = radius_earth * np.cos(lat_rad)
    u = i_cap
    v = j_cap
    omega = 2 * np.pi / 86400  # Earth rotation rad/s
    return [center, radius, u, v, omega]


def create_circular_trajectory_parameters_for_satelites(theta_inclination_rad=0.0, raan_rad=0.0, radius_orbit=r_sat):
    """
    Satellite trajectory. All angles in radians.
    """
    center = np.zeros(3)
    u = i_cap * np.cos(raan_rad) + j_cap * np.sin(raan_rad)
    f = np.cross(k_cap, u)
    v = rotate_vector_around_axis_by_angle(f, u, theta_inclination_rad)
    omega = omega_sat
    radius = radius_orbit
    return [center, radius, u, v, omega]


def find_position_in_space_with_3d_circular_trajectory(center, radius, u, v, omega, time=0, verbose=False):
    angular_position = omega * time  # radians
    if verbose:
        print("Angular position at time =", time, "is", angular_position)
    return center + radius * (u * np.cos(angular_position) + v * np.sin(angular_position))


def rays_are_similar(u, v, theta_threshold_rad, tol=1e-12):
    """
    Compare two vectors. All angles in radians.
    """
    u = np.array(u, float)
    v = np.array(v, float)
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    if nu <= tol or nv <= tol:
        raise ValueError("u or v is zero — cannot define ray direction.")
    dot = np.dot(u, v)
    cos_theta = np.clip(dot / (nu * nv), -1, 1)
    theta = np.arccos(cos_theta)
    return (dot > 0) and (theta < theta_threshold_rad), theta



In [ ]:
file_path = '/content/drive/MyDrive/space_robotics/rectanna_placement.csv'

# Load CSV into a DataFrame
data_rectanna_placement_degrees = pd.read_csv(file_path)

# Optional: view first few rows
data_rectanna_placement_radians=np.radians(data_rectanna_placement_degrees)
data_rectanna_placement_radians

,x,y
0,0.794412,0.661670
1,0.063660,-1.259791
2,0.433655,1.365990
3,1.068314,-2.104112
4,1.113790,2.161475
5,0.457652,0.158797
6,-0.144542,2.436178
7,1.280683,-0.848478
8,0.158490,0.276183
9,1.123878,1.545994


In [ ]:

# --- simulation loop ---
cnt = 0
start = time.time()
total_seconds = 86400 * 6  # iterations
theta_threshold_rad = 10 * np.pi / 180  # convert 10 degrees to radians
lat=20
lng=20
# Make sure lat/long are in radians
lat_rad = lat * np.pi / 180
long_rad = lng * np.pi / 180
params_sats = create_circular_trajectory_parameters_for_satelites(theta_inclination_rad=np.deg2rad(53), raan_rad=0)


for i in tqdm(range(total_seconds)):
    params_rectannas = create_circular_trajectory_parameters_for_rectannas(lat_rad=lat_rad, long_rad=long_rad)

    pos_rectannas = find_position_in_space_with_3d_circular_trajectory(
        center=params_rectannas[0],
        radius=params_rectannas[1],
        u=params_rectannas[2],
        v=params_rectannas[3],
        omega=params_rectannas[4],
        time=i
    )

    pos_sats = find_position_in_space_with_3d_circular_trajectory(
        center=params_sats[0],
        radius=params_sats[1],
        u=params_sats[2],
        v=params_sats[3],
        omega=params_sats[4],
        time=i
    )

    bool_value, theta_value = rays_are_similar(
        pos_rectannas,
        pos_sats,
        theta_threshold_rad
    )

    if bool_value:
        cnt += 1

# --- total time elapsed ---
end = time.time()
elapsed = end - start

def format_hms(s):
    s = int(s)
    h = s // 3600
    m = (s % 3600) // 60
    sec = s % 60
    return f"{h}h {m:02d}m {sec:02d}s"

print(f"Total count: {cnt}, ratio: {cnt/total_seconds:.6f}")
print(f"Total time elapsed: {elapsed:.2f} seconds ({format_hms(elapsed)})")


  0%|          | 0/518400 [00:00<?, ?it/s]

Total count: 3493, ratio: 0.006738
Total time elapsed: 89.28 seconds (0h 01m 29s)


In [ ]:
import time
import numpy as np
from tqdm import tqdm

def calculate_ratio_activation(theta_inclination_deg, raan_deg, total_time_seconds, lat_long_array, theta_threshold_deg=10):
    """
    Simulate satellite and multiple rectenna visibility.

    Args:
        theta_inclination_deg: Satellite inclination in degrees
        raan_deg: Right Ascension of Ascending Node in degrees
        total_time_seconds: Number of seconds (iterations) to simulate
        lat_long_array: np.array of shape (N,2) containing latitudes and longitudes in degrees
        theta_threshold_deg: Threshold angle for ray similarity (default 10 degrees)

    Returns:
        counts: np.array of shape (N,) with counts of times rays are similar for each rectenna
        ratios: np.array of shape (N,) with ratio = count / total_time_seconds
        elapsed_time_sec: total simulation time in seconds
    """
    N = lat_long_array.shape[0]
    counts = np.zeros(N, dtype=int)

    # Convert angles to radians
    theta_inclination_rad = theta_inclination_deg * np.pi / 180
    raan_rad = raan_deg * np.pi / 180
    theta_threshold_rad = theta_threshold_deg * np.pi / 180

    # Precompute satellite parameters
    params_sats = create_circular_trajectory_parameters_for_satelites(
        theta_inclination_rad=theta_inclination_rad,
        raan_rad=raan_rad
    )

    start = time.time()

    for i in tqdm(range(total_time_seconds)):
        for idx in range(N):
            lat_rad = lat_long_array[idx,0] * np.pi / 180
            long_rad = lat_long_array[idx,1] * np.pi / 180

            # Rectenna parameters
            params_rectannas = create_circular_trajectory_parameters_for_rectannas(
                lat_rad=lat_rad,
                long_rad=long_rad
            )

            # Positions
            pos_rectannas = find_position_in_space_with_3d_circular_trajectory(
                center=params_rectannas[0],
                radius=params_rectannas[1],
                u=params_rectannas[2],
                v=params_rectannas[3],
                omega=params_rectannas[4],
                time=i
            )

            pos_sats = find_position_in_space_with_3d_circular_trajectory(
                center=params_sats[0],
                radius=params_sats[1],
                u=params_sats[2],
                v=params_sats[3],
                omega=params_sats[4],
                time=i
            )

            # Ray similarity
            bool_value, theta_value = rays_are_similar(
                pos_rectannas,
                pos_sats,
                theta_threshold_rad
            )

            if bool_value:
                counts[idx] += 1

    elapsed_time_sec = time.time() - start
    ratios = counts / total_time_seconds

    return counts, ratios, elapsed_time_sec

# Optional helper
def format_hms(s):
    s = int(s)
    h = s // 3600
    m = (s % 3600) // 60
    sec = s % 60
    return f"{h}h {m:02d}m {sec:02d}s"


In [ ]:
cnt, ratio, elapsed = calculate_ratio_activation(
    theta_inclination_deg=53,
    raan_deg=0,
    total_time_seconds=86400*1,
    lat_long_array=data_rectanna_placement_radians.values,

)



100%|██████████| 86400/86400 [03:19<00:00, 433.39it/s]


TypeError: unsupported format string passed to numpy.ndarray.__format__

In [ ]:
cnt

array([776, 771, 774, 778, 778, 773, 769, 778, 772, 777, 777, 769, 778,
       767, 763, 777, 776, 773, 774, 777, 778, 767, 779, 764, 775, 777,
       769, 777, 773, 775, 779, 775, 777, 775, 778, 778, 772, 767, 773,
       768, 771, 778, 777, 778, 770, 778, 767, 774, 773, 775])

In [ ]:
theta_sat_values = np.linspace(0, np.pi, 100)

# Divide -pi/2 to pi/2 into 100 parts
raan_sat_values = np.linspace(-np.pi/2, np.pi/2, 100)

# Nested loop
for theta in theta1_values:
    for raan in theta2_values:

